# Notebook 01: Data Loading and Exploratory Data Analysis

**Project:** Classification of High-Protein and Low-Protein Corn (*Zea Mays*) Using NIR Spectral Data and Machine Learning Techniques  
**Author:** Kervin Ralph A. Samson 
**Dataset:** Eigenvector Research Public Corn NIR Dataset  

---

This notebook is the **first** in the pipeline. Its sole purpose is to load the raw dataset as-is, inspect its structure, and visualize both the spectral data and the constituent labels. **No preprocessing, no splitting, and no modifications are applied here.** All downstream transformations are deferred to subsequent notebooks.

The insights gathered here — particularly the protein distribution statistics — directly inform the classification threshold defined in Notebook 02.

## Section 1 — Imports and Setup

We begin by importing every library needed for this notebook. Only `pandas`, `numpy`, `matplotlib`, and `seaborn` are used — no machine learning libraries are loaded at this stage, consistent with the read-only, exploratory purpose of this notebook.

Matplotlib and seaborn are configured for publication-quality output: a clean style, readable font sizes, and high-resolution figures. A global `RANDOM_STATE = 42` is declared for reproducibility in any operation that may involve randomness.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot style ─────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

print('Libraries loaded successfully.')
print(f'RANDOM_STATE = {RANDOM_STATE}')

## Section 2 — Load the Dataset

The dataset used in this study is the **Eigenvector Research public corn NIR dataset**, pre-converted from MATLAB format (`.mat`) to CSV. It contains NIR diffuse reflectance spectra measured on an **mp5** instrument for 80 corn samples, together with wet-chemistry reference values for four constituents: Moisture, Starch, Oil, and Protein.

The CSV is structured as follows:

| Column range | Contents |
|---|---|
| `SampleID` | Sample identifiers S001–S080 |
| `Wave_1`–`Wave_700` | NIR absorbance at 700 wavelengths (1100–2498 nm, 2 nm intervals) |
| `Moisture`, `Starch`, `Oil`, `Protein` | Constituent labels (% dry-weight basis) |

After loading, we partition the dataframe into:
- **`X`** — the spectral matrix (80 × 700), stored as a NumPy array for downstream numerical operations.
- **`labels_df`** — the four constituent columns, kept as a Pandas DataFrame.
- **`sample_ids`** — the sample identifier column, kept as a Python list.

In [ ]:
# ── Load ───────────────────────────────────────────────────────────────────────
DATA_PATH = '../data/raw/corn_mp5_regression_data.csv'
df = pd.read_csv(DATA_PATH)

print(f'DataFrame shape: {df.shape}  (rows × columns)')
print()
df.head()

In [ ]:
# ── Partition ──────────────────────────────────────────────────────────────────
SPECTRAL_COLS  = [f'Wave_{i}' for i in range(1, 701)]
LABEL_COLS     = ['Moisture', 'Starch', 'Oil', 'Protein']

sample_ids = df['SampleID'].tolist()
X          = df[SPECTRAL_COLS].values          # NumPy array  (80 × 700)
labels_df  = df[LABEL_COLS].copy()             # DataFrame    (80 × 4)

print(f'sample_ids : {len(sample_ids)} entries  [{sample_ids[0]} … {sample_ids[-1]}]')
print(f'X shape    : {X.shape}  (samples × wavelength channels)')
print(f'labels_df  : {labels_df.shape}  (samples × constituents)')

## Section 3 — Descriptive Statistics of the Four Constituents

Before any visualization, it is useful to obtain a statistical summary of all four reference constituents. The `describe()` output provides the count, mean, standard deviation, minimum, quartiles, and maximum for each column.

We specifically print the **median protein value**, because this study uses a median split to define the binary classification target: samples with protein content above the median are labeled *High Protein*, and those at or below the median are labeled *Low Protein*. Using the median rather than an arbitrary threshold ensures a balanced class distribution, which is critical for fair model training and evaluation.

> **Note:** Although all four constituents are summarized here for completeness, **only Protein is used as the classification target** in this study. Moisture, Starch, and Oil are included in exploratory plots (Section 7) to contextualize inter-constituent relationships.

In [ ]:
# ── Descriptive statistics ─────────────────────────────────────────────────────
print('Descriptive statistics — all four constituents:\n')
display(labels_df.describe().round(4))

median_protein = labels_df['Protein'].median()
print(f'\nMedian protein content : {median_protein:.4f} %')
print('(This value will serve as the binary classification threshold in Notebook 02.)')

## Section 4 — Protein Distribution Plot

Understanding the distribution of the target variable is a prerequisite for any classification task. The histogram below shows the frequency of protein content values across all 80 samples, overlaid with a kernel density estimate (KDE) for a smooth view of the underlying distribution.

The **dashed red vertical line** marks the median protein value, which will be used as the classification threshold. Plotting this line makes it visually clear that the median split divides the dataset into two roughly equal halves, justifying it as a principled and balanced labeling strategy.  A symmetric, unimodal distribution centred near the median would further support this choice.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.histplot(
    labels_df['Protein'],
    bins=15,
    kde=True,
    color='steelblue',
    edgecolor='white',
    linewidth=0.6,
    ax=ax,
    label='Protein count + KDE'
)

ax.axvline(
    median_protein,
    color='red',
    linestyle='--',
    linewidth=2.0,
    label=f'Median = {median_protein:.3f} %'
)

ax.set_title('Distribution of Protein Content Across 80 Corn Samples', fontsize=14, fontweight='bold')
ax.set_xlabel('Protein Content (%)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.legend()

plt.tight_layout()
plt.show()

## Section 5 — Raw NIR Spectra Plot

Visualizing the raw spectra before any preprocessing is essential to establish a baseline understanding of the data's appearance. Each of the 80 spectra is plotted as a thin, semi-transparent line against the true physical wavelength axis (1100–2498 nm at 2 nm intervals).

Using a low alpha value (`alpha = 0.3`) allows overlapping regions — where many samples share similar absorbance — to appear darker, while more variable regions remain lighter. This gives an intuitive sense of where spectral variability across samples is highest. Notable absorption bands in the NIR region of corn spectra are typically associated with:
- **~1450 nm / ~1940 nm**: O–H stretch overtones (moisture)
- **~2100 nm**: C–H combination bands (starch, oil)
- **~2180 nm**: N–H combination bands (protein)

No preprocessing has been applied; the spectra shown are the raw absorbance values loaded directly from the CSV.

In [ ]:
# ── Wavelength axis ────────────────────────────────────────────────────────────
wavelengths = np.arange(1100, 2499, 2)      # 1100, 1102, …, 2498  → 700 points
assert len(wavelengths) == 700, (
    f'Expected 700 wavelength points, got {len(wavelengths)}'
)

fig, ax = plt.subplots(figsize=(11, 5))

for spectrum in X:
    ax.plot(wavelengths, spectrum, color='steelblue', alpha=0.3, linewidth=0.8)

ax.set_title('Raw NIR Spectra of All 80 Corn Samples', fontsize=14, fontweight='bold')
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Absorbance', fontsize=12)

plt.tight_layout()
plt.show()

print(f'Wavelength range : {wavelengths[0]} – {wavelengths[-1]} nm')
print(f'Step size        : 2 nm')
print(f'Total channels   : {len(wavelengths)}')

## Section 6 — Spectra Colored by Protein Group

To provide a preliminary visual intuition about whether NIR spectra carry discriminative protein information, we re-plot all 80 spectra and color each one according to a **provisional protein group** derived from the median split:

- **Red** — Protein > median (High Protein)
- **Blue** — Protein ≤ median (Low Protein)

> ⚠️ **Important:** This coloring is used **exclusively for visualization purposes in this notebook**. The official binary class labels (`1` = High Protein, `0` = Low Protein) will be formally assigned in **Notebook 02 (02_labeling.ipynb)**, which becomes the authoritative source of ground-truth labels for all downstream processing and model training.

If systematic spectral differences are visible between the two groups (e.g., consistent offsets or crossing patterns in specific wavelength regions), this provides early evidence that NIR spectra can indeed discriminate protein content — a key justification for the classification approach taken in this study.

In [ ]:
# ── Provisional group assignment (visualization only) ──────────────────────────
protein_values   = labels_df['Protein'].values
is_high_protein  = protein_values > median_protein   # boolean array

color_map = {True: 'red', False: 'steelblue'}

fig, ax = plt.subplots(figsize=(11, 5))

for spectrum, high in zip(X, is_high_protein):
    ax.plot(
        wavelengths, spectrum,
        color=color_map[high],
        alpha=0.4,
        linewidth=0.9
    )

# ── Legend ─────────────────────────────────────────────────────────────────────
high_patch = mpatches.Patch(color='red',      label=f'High Protein  (n={is_high_protein.sum()})')
low_patch  = mpatches.Patch(color='steelblue', label=f'Low Protein   (n={(~is_high_protein).sum()})')
ax.legend(handles=[high_patch, low_patch], loc='upper right')

ax.set_title('NIR Spectra Colored by Protein Group (Median Split Preview)', fontsize=14, fontweight='bold')
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Absorbance', fontsize=12)

plt.tight_layout()
plt.show()

print(f'Provisional High Protein samples : {is_high_protein.sum()}')
print(f'Provisional Low Protein samples  : {(~is_high_protein).sum()}')
print('(Official labels will be assigned in Notebook 02.)')

## Section 7 — Correlation of Constituents

In agricultural NIR spectroscopy, the four measured constituents (Moisture, Starch, Oil, and Protein) are often biologically inter-correlated. For example, an increase in protein content typically corresponds to a decrease in starch content due to the fixed total dry-weight composition of the grain.

The pairplot below visualizes all pairwise relationships between the four constituents. Diagonal cells show the univariate distribution of each constituent; off-diagonal cells show scatter plots for each pair.

This analysis serves two purposes:
1. **Justify the choice of Protein as the classification target** — its distribution should exhibit sufficient spread to support a meaningful binary split.
2. **Flag potential confounding** — if protein is highly co-linear with another constituent (e.g., starch), spectral models predicting protein may implicitly learn features of the correlated constituent. This is a known challenge in NIR chemometrics and motivates careful feature analysis in later notebooks.

In [ ]:
pair_grid = sns.pairplot(
    labels_df,
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 'edgecolor': 'none', 's': 40},
    diag_kws={'fill': True},
    corner=False
)

pair_grid.figure.suptitle(
    'Pairwise Relationships Between NIR Constituent Labels',
    y=1.02,
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

print('Pearson correlation matrix:')
display(labels_df.corr(method='pearson').round(4))

## Section 8 — Summary

This notebook completed a full exploratory pass over the raw corn NIR dataset. The key findings are:

| Item | Value |
|---|---|
| **Total samples loaded** | 80 (S001 – S080) |
| **Wavelength channels** | 700 (1100 nm – 2498 nm, 2 nm step) |
| **Protein minimum** | see `labels_df['Protein'].min()` |
| **Protein maximum** | see `labels_df['Protein'].max()` |
| **Protein median** | see `median_protein` variable above |

**Observation on protein distribution:** The protein values form a moderately spread, approximately unimodal distribution centred near the median, supporting the median-split strategy as a principled and balanced way to define the binary classification target.

**Next step:** Notebook `02_labeling.ipynb` will formally assign binary class labels (`1` = High Protein, `0` = Low Protein) based on the median split identified here, producing the labeled dataset that all subsequent preprocessing, feature extraction, and model training notebooks will use.

In [ ]:
# ── Compact summary printout ───────────────────────────────────────────────────
protein_min    = labels_df['Protein'].min()
protein_max    = labels_df['Protein'].max()

print('=' * 55)
print('  NOTEBOOK 01 — DATA LOADING SUMMARY')
print('=' * 55)
print(f'  Total samples loaded       : {len(sample_ids)}')
print(f'  Wavelength channels        : {X.shape[1]}')
print(f'  Wavelength range           : {wavelengths[0]} – {wavelengths[-1]} nm')
print(f'  Protein range              : {protein_min:.3f} – {protein_max:.3f} %')
print(f'  Protein median             : {median_protein:.3f} %')
print(f'  Provisional High Protein   : {is_high_protein.sum()} samples')
print(f'  Provisional Low Protein    : {(~is_high_protein).sum()} samples')
print('=' * 55)
print('  Next: 02_labeling.ipynb — assign binary labels')
print('=' * 55)